# Pocket Validation

**BioPipelines example** — does a co-folded ligand land in a *real, detectable* pocket? CA II + acetazolamide is co-folded with Boltz2 and also taken from its crystal structure; the ground-truth pocket is the residues near the bound ligand. Two template-free predictors (P2Rank and AF2BIND) then propose a pocket on each structure, scored by recall and precision against the ground truth.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[colab]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines import Boltz2, AlphaFold, AF2BIND, P2Rank

with Pipeline(project="Setup", job="InstallTools"):
    Boltz2.install()
    P2Rank.install()
    AlphaFold.install()   # LocalColabFold — provides the JAX/params stack AF2BIND reuses
    AF2BIND.install()
    # Consensus needs no install (runs in the base biopipelines env).

## Cell 3: Pocket Validation Pipeline

The motivating question: when you co-fold a (possibly de novo) protein with a ligand, does the ligand actually sit in a pocket a pocket-predictor would find? We test this on carbonic anhydrase II + acetazolamide, on **two** structures:

- the **Boltz2-predicted** complex (the de-novo-style case), sampled as **5 diffusion poses**, and
- the **crystal** complex (PDB 3HS4) as an experimental cross-check.

1. **DistanceSelector** marks, per pose, the residues within 5 Å of the bound ligand (code `AZM`).
2. **Consensus** collapses the 5 predicted poses into ONE robust ground-truth pocket: residues that sit within 5 Å of the dye in **≥50% of poses** (`fraction("distance<=5") >= 0.5`). This avoids trusting a single pose — the pocket is what the model places consistently. The crystal (one pose) uses its `within` set directly.
3. **P2Rank** (surface ML) and **AF2BIND** (AlphaFold2 pair representation) each propose pocket residues on the top predicted pose and on the crystal — P2Rank with `probability > 0.5`, AF2BIND with `p_bind > 0.5`.
4. The **Selection** tool's set algebra gives, per predictor, |pred| and the overlap with the consensus truth.
5. **Panda** computes **recall** = overlap/|truth| and **precision** = overlap/|pred|.
6. A grouped **Plot** compares both predictors on both structures.

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines import (PDB, Ligand, Boltz2, Pool, DistanceSelector, Consensus,
                          Selection, P2Rank, AF2BIND, Panda, Plot)

with Pipeline(project="PocketValidation", job="LigandPocketRecovery"):
    Resources(gpu="A100", time="2:00:00", memory="16GB")

    azm = Ligand("acetazolamide", codes="AZM")     # shared 3-letter code AZM
    ca2 = Sequence("MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSVSYDQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPPLLECVTWIVLKEPISVSSEQVLKFRKLNFNGEGEPEELMVDNWRPAQPLKNRQIKASFK",
                   ids="CA2")

    # --- Predicted complex: 5 diffusion poses, consensus ground-truth pocket. ---
    poses = Boltz2(proteins=ca2, ligands=azm, diffusion_samples=5, top_only=False)
    pose_pockets = DistanceSelector(structures=poses, ligand=poses, distance=5.0)
    # Residues within 5 A of AZM in >=50% of poses -> robust ground truth.
    consensus = Consensus(pose_pockets.streams.distances, groups=ca2,
                          operations=[Consensus.fraction("distance<=5", name="frequency")])
    # Top pose: the single representative the predictors run on.
    pred_top = Boltz2(proteins=ca2, ligands=azm, msas=poses)

    # --- Crystal complex (experimental cross-check), single pose. ---
    crystal = PDB("3HS4", chain="A")
    cryst_pocket = DistanceSelector(structures=crystal, ligand=crystal, distance=5.0)

    # Both representative structures in one stream so predictors run once each.
    structures = Pool(runs=[PDB(pred_top), crystal], streams=["structures"])
    p2rank  = P2Rank(structures=structures)
    af2bind = AF2BIND(structures=structures, chain="A")

    # Ground-truth pocket per structure: consensus (predicted) ∪ within (crystal).
    # Both are resi-csv/selection inputs to Selection, matched per-structure by id.
    truth = Selection(Selection.add(consensus.streams.distances, include="frequency>=0.5"),
                      Selection.add(cryst_pocket.tables.selections.within),
                      structures=structures)

    # Predicted pocket sets, and the truth residues each predictor misses.
    p2_pred = Selection(Selection.add(p2rank.streams.residues, include="probability>0.5"),
                        structures=structures)
    p2_miss = Selection(Selection.add(consensus.streams.distances, include="frequency>=0.5"),
                        Selection.add(cryst_pocket.tables.selections.within),
                        Selection.subtract(p2rank.streams.residues, include="probability>0.5"),
                        structures=structures)
    af_pred = Selection(Selection.add(af2bind.streams.binding, include="p_bind>0.5"),
                        structures=structures)
    af_miss = Selection(Selection.add(consensus.streams.distances, include="frequency>=0.5"),
                        Selection.add(cryst_pocket.tables.selections.within),
                        Selection.subtract(af2bind.streams.binding, include="p_bind>0.5"),
                        structures=structures)

    # Recall = overlap/|truth|, Precision = overlap/|pred|; overlap = |truth| - |miss|.
    scores = Panda(
        tables=[truth.tables.selections, p2_pred.tables.selections, p2_miss.tables.selections,
                af_pred.tables.selections, af_miss.tables.selections],
        operations=[
            Panda.merge(prefixes=["truth_", "p2pred_", "p2miss_", "afpred_", "afmiss_"]),
            Panda.calculate({
                "p2rank_recall":    "(truth_n_residues - p2miss_n_residues) / truth_n_residues",
                "p2rank_precision": "(truth_n_residues - p2miss_n_residues) / p2pred_n_residues",
                "af2bind_recall":   "(truth_n_residues - afmiss_n_residues) / truth_n_residues",
                "af2bind_precision":"(truth_n_residues - afmiss_n_residues) / afpred_n_residues"}),
        ])

    Plot(Plot.Bar(data=scores.tables.result,
                  x="id",
                  y="p2rank_recall", y_right="af2bind_recall",
                  title="Pocket recall vs consensus truth: P2Rank vs AF2BIND (Boltz2 vs crystal)",
                  xlabel="Structure", ylabel="P2Rank recall", ylabel_right="AF2BIND recall"))
scores.tables.result